In [0]:
# Databricks notebook source
# MAGIC 

In [0]:
%run ./shared_audit_utils

DataFrame[]

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import *

import logging
from datetime import datetime

In [0]:
# Logger
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s"
)

logger = logging.getLogger(
    "bronze_pipeline"
)

In [0]:
# Configuration
PIPELINE_NAME = "online_retail_bronze"

config = (
    spark.table("workspace.default.pipeline_config")
    .filter(f"pipeline_name = '{PIPELINE_NAME}'")
    .first()
)

if config is None:
    raise Exception(
        f"Pipeline config not found: {PIPELINE_NAME}"
    )

SOURCE_FILE = config.source_name
TARGET_TABLE = config.target_table
SOURCE_PATH = (
    f"/Volumes/workspace/default/landing/{SOURCE_FILE}"
)
logger.info(
    f"Source Path: {SOURCE_PATH}"
)

2026-06-06 19:51:17,045 INFO Source Path: /Volumes/workspace/default/landing/Online Retail.csv


In [0]:
# Step 1: Read source file
def read_source() -> DataFrame:

    logger.info(
        f"Reading source file: {SOURCE_FILE}"
    )

    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(SOURCE_PATH)
    )

In [0]:
# Source Validation
def validate_source(df: DataFrame):
    logger.info("Validating source file")
    if df.limit(1).count() == 0:

        raise Exception(
            "Source file is empty"
        )

In [0]:
# Add Metadata

def add_metadata(df: DataFrame) -> DataFrame:

    logger.info("Adding metadata columns")

    return (
        df
        .withColumn("load_timestamp",current_timestamp())
        .withColumn("pipeline_name",lit(PIPELINE_NAME))
    )

In [0]:
# Step 6: Log Data Quality Metrics
def log_data_quality(df: DataFrame):

    null_customer = (
        df.filter(col("CustomerID").isNull()).count()
    )

    negative_quantity = (
        df.filter(col("Quantity") < 0).count()
    )

    null_description = (
        df.filter(col("Description").isNull()).count()
    )

    cancelled_orders = (
        df.filter(col("InvoiceNo").startswith("C")).count()
    )

    logger.info(f"Null CustomerID: {null_customer}")

    logger.info(f"Negative Quantity: {negative_quantity}")

    logger.info(f"Null Description: {null_description}")

    logger.info(f"Cancelled Orders: {cancelled_orders}")

2026-06-06 19:51:17,939 INFO Received command c on object id p0


In [0]:
# Write Bronze
def write_bronze(df: DataFrame):

    logger.info(f"Writing table: {TARGET_TABLE}")

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(
            TARGET_TABLE
        )
    )

In [0]:
# Verify write
def verify_write():

    count = (
        spark.table(TARGET_TABLE).count()
    )

    logger.info(f"Bronze Count: {count}")

    return count

2026-06-06 19:51:18,398 INFO Received command c on object id p0


In [0]:
# Main
def main(run_id,start_time):

    logger.info("Starting bronze pipeline")

    source_df = read_source()

    validate_source(
        source_df
    )

    input_count = (
        source_df.count()
    )

    bronze_df = (
        source_df.transform(add_metadata)
    )

    log_data_quality(
        bronze_df
    )

    write_bronze(
        bronze_df
    )

    output_count = (
        verify_write()
    )

    end_time = datetime.now()
    # logger.info("Writing logs to audit table")
    write_audit_record(
        run_id=run_id,
        pipeline_name=PIPELINE_NAME,
        start_time=start_time,
        end_time=end_time,
        status="SUCCESS",
        input_count=input_count,
        output_count=output_count,
        error_message=None
    )

    logger.info(
        "Bronze pipeline completed"
    )

In [0]:
# Entry Point
if __name__ == "__main__":

    run_id = get_run_id()
    start_time = datetime.now()

    try:

        main(
            run_id,
            start_time
        )

    except Exception as e:

        end_time = datetime.now()

        write_audit_record(
            run_id=run_id,
            pipeline_name=PIPELINE_NAME,
            start_time=start_time,
            end_time=end_time,
            status="FAILED",
            input_count=0,
            output_count=0,
            error_message=str(e)
        )

        logger.exception(
            f"Pipeline {PIPELINE_NAME} failed: {e}"
        )

        raise

2026-06-06 19:52:19,046 INFO Starting bronze pipeline
2026-06-06 19:52:19,047 INFO Reading source file: Online Retail.csv
2026-06-06 19:52:19,047 INFO Validating source file
2026-06-06 19:52:24,981 INFO Adding metadata columns
2026-06-06 19:52:28,115 INFO Null CustomerID: 135080
2026-06-06 19:52:28,115 INFO Negative Quantity: 10624
2026-06-06 19:52:28,116 INFO Null Description: 1454
2026-06-06 19:52:28,117 INFO Cancelled Orders: 9288
2026-06-06 19:52:28,117 INFO Writing table: workspace.default.bronze_online_retail
2026-06-06 19:52:33,327 INFO Bronze Count: 541909
2026-06-06 19:52:33,362 INFO Writing logs to audit table
2026-06-06 19:52:36,784 INFO Bronze pipeline completed
